### 8. Verdict Synthesizer Development

#### 1. Objetivo

El objetivo de este componente es generar un veredicto final a nivel de noticia a partir de los resultados obtenidos para las distintas claims extraídas previamente.

Cada claim habrá sido procesada por el flujo de verificación factual compuesto por el Research Agent, el Live RAG y el Evidence Verifier.

El Verdict Synthesizer `no realiza nuevas búsquedas` ni evalúa directamente las fuentes originales. Su función es integrar los veredictos y explicaciones obtenidos para las distintas claims y producir una conclusión global sobre la noticia.

En las noticias en inglés, el sistema podrá incorporar además la señal auxiliar proporcionada por el clasificador de aprendizaje automático. Esta señal **no sustituye** a la evidencia factual y tendrá un papel secundario respecto a los resultados obtenidos mediante verificación externa.

### 2. Entrada del Verdict Synthesizer

El Verdict Synthesizer opera sobre los resultados obtenidos previamente por el
`Evidence Verifier` para cada una de las claims extraídas de la noticia.

Para cada claim se conserva:

- el texto de la afirmación;
- el veredicto factual obtenido;
- si la evidencia recuperada se considera suficiente;
- la explicación generada por el Evidence Verifier.

Las evidencias individuales no se vuelven a proporcionar al sintetizador, ya
que su interpretación corresponde al Evidence Verifier. De esta forma se
mantiene una separación clara de responsabilidades entre la recuperación de
evidencias, la verificación factual y la síntesis final de la noticia.

### 3. Síntesis del veredicto factual

A partir de los resultados obtenidos para cada claim, el Verdict Synthesizer
genera una **conclusión global** sobre la noticia.

El componente no realiza nuevas búsquedas ni vuelve a analizar las evidencias
originales. Su entrada está formada exclusivamente por los resultados producidos
por el Evidence Verifier.

La síntesis considera el veredicto de cada claim, la suficiencia de la evidencia
y la explicación asociada, permitiendo representar situaciones en las que una
noticia contiene afirmaciones con resultados distintos.

#### 3.1. Prueba de síntesis con múltiples claims

Se construye un caso de prueba compuesto por varias claims con resultados de verificación diferentes.

El objetivo es comprobar que el Verdict Synthesizer integra correctamente los veredictos individuales, la suficiencia de evidencia y las explicaciones generadas por el Evidence Verifier, sin realizar nuevas búsquedas ni reinterpretar directamente las fuentes originales.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI()

In [16]:
from src.verdict_synthesizer import (
    ClaimVerificationSummary,
    synthesize_verdict,
)

In [4]:
claim_summaries = [
    ClaimVerificationSummary(
        claim="La Unión Europea prohibirá completamente los pagos en efectivo a partir de 2027.",
        verdict="REFUTED",
        evidence_sufficient=True,
        explanation=(
            "La evidencia muestra que se establece un límite de 10.000 euros "
            "para determinados pagos en efectivo, no una prohibición completa."
        ),
    ),
    ClaimVerificationSummary(
        claim="La nueva normativa europea comenzará a aplicarse en julio de 2027.",
        verdict="SUPPORTED",
        evidence_sufficient=True,
        explanation=(
            "Las fuentes indican que el Reglamento será aplicable "
            "a partir del 10 de julio de 2027."
        ),
    ),
    ClaimVerificationSummary(
        claim="La medida eliminará completamente el uso del efectivo en todos los Estados miembros.",
        verdict="REFUTED",
        evidence_sufficient=True,
        explanation=(
            "Las fuentes describen restricciones para determinados pagos, "
            "pero no la eliminación general del efectivo."
        ),
    ),
]


In [5]:
final_verdict = synthesize_verdict(
    claim_summaries=claim_summaries,
    client=client,
)

final_verdict

FinalVerdict(verdict='REFUTED', explanation='El artículo contiene una afirmación central falsa: la UE no prohibirá completamente los pagos en efectivo ni eliminará su uso en todos los Estados miembros. La normativa establece límites para determinados pagos en efectivo, incluido un umbral de 10.000 euros. Aunque es correcto que el Reglamento será aplicable desde el 10 de julio de 2027, esa fecha no respalda la afirmación de una prohibición total.')

In [10]:
print("Veredicto Final:", final_verdict.verdict)
print("Explicacion:", final_verdict.explanation)

Veredicto Final: REFUTED
Explicacion: El artículo contiene una afirmación central falsa: la UE no prohibirá completamente los pagos en efectivo ni eliminará su uso en todos los Estados miembros. La normativa establece límites para determinados pagos en efectivo, incluido un umbral de 10.000 euros. Aunque es correcto que el Reglamento será aplicable desde el 10 de julio de 2027, esa fecha no respalda la afirmación de una prohibición total.


#### 3.1.1. Resultado de la prueba de síntesis

El Verdict Synthesizer clasifica el conjunto de claims como `REFUTED`.

La explicación generada distingue correctamente entre los distintos resultados
individuales. Aunque una de las claims, relativa a la fecha de aplicación de la
normativa, está respaldada por la evidencia, las afirmaciones centrales sobre
una prohibición completa del efectivo resultan refutadas.

Este comportamiento muestra que la síntesis no se basa únicamente en un conteo
de etiquetas, sino que considera el contenido y la relevancia de las distintas
claims verificadas para producir una conclusión global coherente.

### 4. Integración de la señal auxiliar del modelo ML

Para noticias en inglés, el Verdict Synthesizer puede recibir adicionalmente la
predicción generada por el clasificador de aprendizaje automático.

Esta señal se interpreta únicamente como información auxiliar basada en las
características textuales del artículo completo.

La evidencia factual obtenida mediante el proceso de verificación de claims
mantiene prioridad sobre la predicción del clasificador. Por tanto, la señal ML
no puede sustituir ni contradecir por sí sola una conclusión factual respaldada
por evidencia suficiente.

El `decision_score` del clasificador se conserva como salida del modelo, pero no
se interpreta como una probabilidad.

#### 4.1. Prueba de prioridad de la evidencia factual frente al ML

Se realiza una prueba controlada en la que el resultado factual de las claims y la señal del clasificador de Machine Learning son contradictorios.

El objetivo es comprobar que el Verdict Synthesizer mantiene la evidencia factual como fuente principal para el veredicto final y utiliza la predicción del modelo ML únicamente como información auxiliar.

In [14]:
claim_summaries_en = [
    ClaimVerificationSummary(
        claim="The European Union will completely ban cash payments from 2027.",
        verdict="REFUTED",
        evidence_sufficient=True,
        explanation=(
            "The evidence shows that the EU will introduce a €10,000 limit "
            "for certain cash payments, rather than a complete ban."
        ),
    ),
    ClaimVerificationSummary(
        claim="The new regulation will apply from July 2027.",
        verdict="SUPPORTED",
        evidence_sufficient=True,
        explanation=(
            "The available evidence indicates that the regulation will apply "
            "from 10 July 2027."
        ),
    ),
]

In [17]:
ml_signal = {
    "prediction": 1,
    "label": "TRUE",
    "decision_score": 1.35,
}

final_verdict_with_ml = synthesize_verdict(
    claim_summaries=claim_summaries_en,
    client=client,
    language = 'en',
    ml_signal=ml_signal,
)

final_verdict_with_ml

FinalVerdict(verdict='REFUTED', explanation='The article’s central claim that the EU will completely ban cash payments from 2027 is refuted by sufficient evidence: the regulation establishes a €10,000 cap for certain cash payments, not a blanket ban. Although the stated July 2027 start date is supported (10 July 2027), that correct timing does not make the main claim accurate. The auxiliary ML signal does not override the claim-level evidence.')

##### 4.1.1. Resultado de la prueba de prioridad factual

El Verdict Synthesizer genera un veredicto final `REFUTED` pese a que la señal auxiliar del modelo de Machine Learning indica una predicción favorable a la veracidad del artículo.

La explicación final mantiene como criterio principal los resultados de verificación factual de las claims. En concreto, distingue entre la afirmación principal sobre una prohibición total del efectivo, que resulta refutada, y la referencia temporal a julio de 2027, que sí está respaldada.

Además, el componente indica explícitamente que la señal del clasificador ML no sustituye ni prevalece sobre la evidencia factual disponible.

Este resultado confirma que la integración mantiene la jerarquía definida para el sistema: la verificación basada en evidencia constituye la fuente principal para el veredicto final, mientras que la predicción ML actúa únicamente como señal auxiliar.

Vamos a realizar una segunda prueba para comprobar que no hemos tenido 'suerte' en este caso y que realmente synthesize_verdict funciona. 

In [18]:
claim_summaries_supported = [
    ClaimVerificationSummary(
        claim="The regulation will apply from 10 July 2027.",
        verdict="SUPPORTED",
        evidence_sufficient=True,
        explanation=(
            "Multiple sources confirm that the regulation "
            "will apply from 10 July 2027."
        ),
    ),
    ClaimVerificationSummary(
        claim="The EU will introduce a €10,000 ceiling for certain cash payments.",
        verdict="SUPPORTED",
        evidence_sufficient=True,
        explanation=(
            "The available evidence consistently confirms "
            "the €10,000 cash-payment limit."
        ),
    ),
]

ml_signal_fake = {
    "prediction": 0,
    "label": "FAKE",
    "decision_score": -1.42,
}

final_verdict_supported = synthesize_verdict(
    claim_summaries=claim_summaries_supported,
    client=client,
    language="en",
    ml_signal=ml_signal_fake,
)

final_verdict_supported

FinalVerdict(verdict='SUPPORTED', explanation='Both verified claims are supported by sufficient evidence: the regulation applies from 10 July 2027 and introduces a €10,000 ceiling for certain cash payments in the EU. Although the auxiliary ML model labels the article as fake, that signal is secondary and does not outweigh the claim-level factual verification.')

##### 4.1.2. Prueba inversa de prioridad factual

Se realiza una segunda prueba, como hemos dicho, en la que `todas las claims` están respaldadas por evidencia suficiente y el clasificador de Machine Learning produce una señal contradictoria, etiquetando el artículo como `FAKE`.

El Verdict Synthesizer devuelve un resultado final `SUPPORTED`.

La explicación indica que las claims verificadas están respaldadas por evidencia suficiente y que la señal auxiliar del modelo ML **no prevalece** sobre los resultados obtenidos mediante verificación factual.

`Esta segunda prueba confirma que la jerarquía definida para el sistema se mantiene en ambos sentidos: una predicción ML favorable no puede anular evidencia factual que refuta una noticia, y una predicción ML desfavorable tampoco puede invalidar claims respaldadas por evidencia suficiente`